# 02 · Limpieza y transformación

Aplicamos las decisiones del diario de hallazgos (notebook 01) a `bank-additional.csv` y `customer-details.xlsx`. Los datos limpios se guardan en `data/processed/`.

In [1]:
# Cargas de nuevo los datos.

import pandas as pd

ruta_bank = "../data/raw/bank-additional.csv"
ruta_customer = "../data/raw/customer-details.xlsx"

df_bank = pd.read_csv(ruta_bank)
hojas_customer = pd.read_excel(ruta_customer, sheet_name=None)

print(df_bank.shape)
for nombre, hoja in hojas_customer.items():
    print(nombre, hoja.shape)



(43000, 24)
2012 (20115, 7)
2013 (8965, 7)
2014 (14090, 7)


In [2]:
# Eliminar columnas sin valor para el análisis (ver diario de hallazgos)

cols_eliminar = ["Unnamed: 0", "latitude", "longitude"]

df_bank = df_bank.drop(columns=cols_eliminar)
print(df_bank.shape)

(43000, 21)


In [3]:
# Pasar todas las columnas de texto a minúsculas

cols_texto = ["job", "marital", "education", "contact", "poutcome", "y"]

for col in cols_texto:
    df_bank[col] = df_bank[col].str.lower()

In [4]:
# job: quitar el punto de "admin."

df_bank["job"] = df_bank["job"].str.replace(".", "", regex=False)

# education: cambiar el punto separador por guion bajo

df_bank["education"] = df_bank["education"].str.replace(".", "_", regex=False)

In [5]:
# Comprobar que los cambios se hicieron correctamente

for col in cols_texto:
    print(df_bank[col].value_counts(dropna=False))
    print("-" * 40)

job
admin            10873
blue-collar       9654
technician        7026
services          4162
management        3050
retired           1790
entrepreneur      1522
self-employed     1489
housemaid         1123
unemployed        1063
student            903
NaN                345
Name: count, dtype: int64
----------------------------------------
marital
married     25999
single      12105
divorced     4811
NaN            85
Name: count, dtype: int64
----------------------------------------
education
university_degree      12722
high_school             9925
basic_9y                6309
professional_course     5477
basic_4y                4356
basic_6y                2386
NaN                     1807
illiterate                18
Name: count, dtype: int64
----------------------------------------
contact
cellular     27396
telephone    15604
Name: count, dtype: int64
----------------------------------------
poutcome
nonexistent    37103
failure         4461
success         1436
Name: count,

In [6]:
# Columnas con coma decimal: cambiar "," por "." y pasar a float
cols_coma = ["cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]

for col in cols_coma:
    df_bank[col] = df_bank[col].str.replace(",", ".", regex=False).astype(float)

print(df_bank[cols_coma].dtypes)



cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
dtype: object


In [7]:
# Comprobar que los valores son razonables
print(df_bank[cols_coma].describe().round(3))

       cons.price.idx  cons.conf.idx  euribor3m  nr.employed
count       42529.000      43000.000  33744.000    43000.000
mean           93.574        -40.509      3.617     5166.853
std             0.580          4.637      1.737       72.411
min            92.201        -50.800      0.634     4963.600
25%            93.075        -42.700      1.344     5099.100
50%            93.749        -41.800      4.857     5191.000
75%            93.994        -36.400      4.961     5228.100
max            94.767        -26.900      5.045     5228.100


In [8]:
# Separar la fecha por "-" y quedarnos con la parte del mes
meses_en_datos = df_bank["date"].str.split("-", expand=True)[1]
print(meses_en_datos.value_counts(dropna=False))    

1
noviembre     3603
octubre       3599
julio         3593
marzo         3589
abril         3582
febrero       3577
septiembre    3569
mayo          3547
junio         3538
agosto        3526
enero         3517
diciembre     3512
NaN            248
Name: count, dtype: int64


In [9]:
# Diccionario de traducción: mes en español a número de mes
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

fechas = df_bank["date"]
for mes, numero in meses.items():
    fechas = fechas.str.replace(mes, str(numero), regex=False)

print(fechas.head())

0      2-8-2019
1     14-9-2016
2     15-2-2019
3    29-11-2015
4     29-1-2017
Name: date, dtype: str


In [10]:
# Convertir a formato fecha
df_bank["date"] = pd.to_datetime(fechas, format="%d-%m-%Y")

print(df_bank["date"].dtype)
print(df_bank["date"].min(), "→", df_bank["date"].max())
print("Nulos:", df_bank["date"].isnull().sum())


datetime64[us]
2015-01-01 00:00:00 → 2019-12-31 00:00:00
Nulos: 248


In [11]:
# Nuevas columnas a partir de la fecha (las pide el enunciado)
df_bank["contact_month"] = df_bank["date"].dt.month.astype("Int64")
df_bank["contact_year"] = df_bank["date"].dt.year.astype("Int64")

print(df_bank[["date", "contact_month", "contact_year"]].head())
print(df_bank["contact_year"].value_counts(dropna=False).sort_index())

        date  contact_month  contact_year
0 2019-08-02              8          2019
1 2016-09-14              9          2016
2 2019-02-15              2          2019
3 2015-11-29             11          2015
4 2017-01-29              1          2017
contact_year
2015    8544
2016    8533
2017    8562
2018    8549
2019    8564
<NA>     248
Name: count, dtype: Int64
